# Daily Cross-Sectional Regression — Construction Spec (USE4-faithful)

**Purpose.** This notebook is the **source-of-truth spec** for building
`daily_csr` — the **Daily** sibling of the monthly CSR
(`csr_spec.ipynb`). One engine, two clocks: the same constrained WLS, run once
per **trading day** with the prevailing month-end exposures held fixed
intramonth. This stage is **Layer 0 of the risk model**: the covariance
forecast (`06_fcov`) is estimated from its daily factor returns, and the
specific-risk model (`07_specific_risk`) from its daily ESTU residuals —
~6,350 daily observations where the monthly CSR has ~303.

**Audience.** You — plus whatever AI assistant you are driving. Treat its
output as deeply untrustworthy until audited. Follow the stages linearly.

✅ **PDF SPEC (USE4 Methodology Notes; Empirical Notes).** USE4 estimates
factor returns from **daily** cross-sectional regressions; the factor
covariance matrix (Newey–West, eigenfactor adjustment, volatility-regime
adjustment) and the specific-risk model are built from the resulting daily
factor and specific returns. Exposures update on the monthly model calendar —
intramonth, the regression reuses the latest month-end exposure matrix. The
regression itself — identification, √mcap WLS weights, the cap-weighted
industry constraint, the exact reparametrization — is specified once, in
`csr_spec.ipynb` §4; this spec does not restate it.

**The model.** For trading day d owned by month-end t, with n complete-case
ESTU stocks:

```
r_d = X_t f_d + u_d        X_t = [1 | D | S] frozen at t        v ∝ √mcap at t
```

subject to `Σ_j w_jt · f_ind,j,d = 0` (constraint weights also frozen at t).
Only the left-hand side is daily.

**Strict PIT.** Day d's return is explained by the most recent month-end
**strictly before** d (previous-trading-day convention, §2) — exposures always
precede the return they explain.

> **Reference.** The monthly CSR chapter in `textbook/` carries the reference
> numbers; the daily sibling has no chapter of its own. Its §6 validation
> contract — above all, reconciliation to the monthly factor returns — is the
> evidence gate. Treat it as non-negotiable.

> **Your task.** Write `daily_csr_build.ipynb`. Run it **immediately after
> `csr_build.ipynb`** (§6 reconciles against the monthly factor returns) and
> **before** `06_fcov` and `07_specific_risk`, which consume its two outputs.

## §1. Deliverables and output schema

Both files under `data/out/`, compression zstd, statistics=True.

**Daily factor returns:** `data/out/csr_factor_returns_daily.parquet`

| Column | Type | Description |
|---|---|---|
| `date` | Date | The trading day the return is realized on |
| `factor` | String | `country`, the 55 industry names, or the 12 style score names |
| `factor_type` | String | `country` / `industry` / `style` (grouping convenience) |
| `f` | Float64 | Daily factor return (null if the column was absent or degenerate that day) |
| `r2` | Float64 | Weighted cross-sectional R² of the day's regression (repeated per row) |
| `n_stocks` | UInt32 | Stocks in the day's regression (repeated per row) |

68 rows per solved day (1 + 55 + 12). Note the **single `date` column** — the
monthly file's `signal_date`/`ret_date` pair collapses to the return day; the
owning month-end is recoverable via the §2 ownership mapping (master list #7).
Reference run: 431,800 rows over 6,350 days, 3.3 MB.

**Daily specific returns:** `data/out/csr_specific_returns_daily.parquet`

| Column | Type | Description |
|---|---|---|
| `permaticker` | Int64 | Sharadar permanent ticker ID |
| `date` | Date | Trading day |
| `u` | Float64 | Daily specific return — the ESTU regression residual `y − X f` |

**ESTU rows only** — no non-ESTU out-of-sample extension (§5, master list #5)
— and **sorted `[permaticker, date]`** so per-stock time-series scans are
contiguous. This file is Layer 0 of the specific-risk build. Reference run:
13,379,774 stock-days, 6,492 tickers, 6,350 days, 107.0 MB.

Reference-run counts are data-vintage-dependent (data through 2026-06 here);
your rebuild will drift in tail rows, not in shape.

## §2. Pipeline

```
STAGE 0  Setup, parameters
STAGE 1  Anchor: read country_use4.parquet (regression universe + normalized
         √mcap weights); assert in-ESTU only, country ≡ 1.0, Σ w_reg = 1 per
         date (to 1e-12)
STAGE 2  Exposure matrix: join the 12 style scores + industry labels onto the
         anchor (assert row count unchanged, no null industry); load the
         industry constraint weights (assert exactly 55 industries)
STAGE 3  Daily returns + ownership: load the shared daily panel you built in
         01.5_daily (data/out/daily_returns.parquet); map every trading day
         to its owning month-end (previous-trading-day convention)
STAGE 4  Daily constrained WLS: per trading day, the monthly CSR's
         reparametrized engine unchanged; collect f, R², n, and the ESTU
         residual vector u
STAGE 5  Ship: csr_factor_returns_daily.parquet +
         csr_specific_returns_daily.parquet (read-back row counts asserted)
STAGE 6  Validate (four checks, §6)
```

**Ownership mapping (strict PIT).** Build the trading-day calendar; for each
day d take its **previous trading day**; as-of join (backward) that previous
day onto the monthly rebalance calendar. The matched month-end t **owns** d.
Consequences:

- t < d always — exposures strictly precede the return they explain.
- The days owned by t are **exactly** the days the monthly CSR sums into its
  y for transition (t, t+1]. The two builds partition the calendar
  identically — that is what makes the §6 reconciliation check meaningful.
- Days before the first month-end (the pre-1999 warm-up) own nothing and are
  dropped.

**Inputs** (all `data/out/`): `country_use4.parquet` (anchor), the 12
`<slug>_use4.parquet` style files, `industries_use4.parquet` (industry
labels only), `industry_weights_use4.parquet` (constraint weights),
`daily_returns.parquet` (daily excess log returns + benchmark `mkt_ret`),
and — for validation only — `csr_factor_returns.parquet` from
`csr_build.ipynb`.

**Position in the pipeline:** `04 country → 05 csr_build → 05 daily_csr_build
→ 06 fcov_build → 07 specific_risk_build → 08 risk_decomp_build`. Run this
build immediately after the monthly CSR; both `06_fcov` (daily factor
returns) and `07_specific_risk` (daily residuals) consume its outputs.

Reference-run sanity prints: anchor 822,944 rows / 330 month-ends
(1999-01-29 → 2026-06-30); 180,185 stock-dates carry ≥ 1 missing style score
(complete-case drops, §3); joined daily panel ≈ 36.9M stock-days over 6,874
candidate trading days.

## §3. STAGE 0 — Parameters

```python
# ───── Parameters ─────────────────────────────────────────────────────────────
# ✅ PDF SPEC — engine constants, identical to the monthly CSR (csr_spec.ipynb §3):
#    country exposure is definitional; WLS weights ∝ √mcap; constraint:
#    cap-weighted industry factor returns sum to zero per date.
COUNTRY_EXPOSURE = 1.0

# 💡 DEFAULT (NOT IN PDF) — calendar and sample rules (same values as monthly)
CALENDAR_START   = date(1999, 1, 1)
MIN_SAMPLE       = 100     # complete-case stocks per day; below -> skip the day
UNASSIGNED_LABEL = "UNASSIGNED (fallback ladder)"
UNASSIGNED_MAX   = 50

# 💡 DEFAULT (NOT IN PDF) — daily validation gates (§6). Corr gates are looser
#    than the monthly 0.99: single-day cross-sections are noisier, and the
#    reconciliation target is a differently-sampled regression (§6 note 2).
FC_CORR_MIN    = 0.95      # corr(daily f_c, daily ESTU market excess return)
RECONCILE_MIN  = 0.95      # corr(Σ intramonth daily f, monthly CSR f)
CONSTRAINT_TOL = 1e-8      # max |Σ_j w_j·f_ind,j| per day (monthly gates 1e-10)
COND_MAX       = 1e6       # max condition number of the scaled design
MIN_DAYS       = 6000      # trading-day coverage floor

STYLES = {"size": "size_score", "bp": "btop_score", "dyld": "dyld_score",
          "eyld": "eyld_score", "gro": "gro_score", "lev": "lev_score",
          "liq": "liq_score", "beta": "beta_score", "mom": "mom_score",
          "resvol": "resvol_score", "nlb": "nlb_score", "nls": "nls_score"}
```

❓ **NOT IN PDF — exact reconciliation tolerance.** The reference build also
defined `RECONCILE_TOL = 1e-9` (sum-of-daily f vs the monthly f) but never
used it anywhere.

💡 **DEFAULT — dropped.** Exact reconciliation is impossible by construction:
the monthly CSR is **one** regression on month-summed returns over one pooled
complete-case sample, while the daily sum stacks ~21 regressions whose
samples differ day to day (stocks trade, list, and delist intramonth). The
reconciliation check is therefore statistical — corr > 0.95 with the RMSE
reported — not exact (master list #8). Do not carry a dead constant.

**No daily R² gate.** Mean daily R² is informational only (the monthly spec
gates its mean at 0.15); expect ≈ 0.25 at either frequency — reference run
0.253 daily vs 0.252 monthly.

**Complete-case regression.** The daily CSR estimates nothing on the exposure
side — it consumes the month-end exposure matrix as-is. Stock-days missing
any of the 12 style scores are dropped from that day's regression, exactly as
in the monthly build. If you later choose to impute missing exposures
upstream, the engine, weights, and constraint are unchanged — only the
per-day sample grows.

## §4. STAGE 4 — The daily engine

**One engine, two clocks.** The regression is the monthly CSR's, verbatim:
identification via the cap-weighted industry constraint, exact
reparametrization (eliminate the largest-cap-weight **present** industry e;
transform `D̃_j = D_j − (w_j/w_e)·D_e`; solve the reduced `[1 | D̃ | S]` by
√v-scaled `lstsq`; recover `f_e = −(Σ_{j≠e} w_j f_j)/w_e`). **Do not
re-derive it and do not fork it** — full algebra and rationale live in
`csr_spec.ipynb` §4. This section specs only what the daily clock changes.

Per trading day d owned by month-end t:

1. **Sample.** Inner-join the exposure matrix at t with day-d realized
   returns; drop rows missing any of the 12 style scores (complete-case);
   drop `UNASSIGNED (fallback ladder)`. If n < `MIN_SAMPLE` → **skip the
   day** and count it.
2. **Target.** y = the single-day excess log return. No compounding — the
   monthly build's owned-days summation happens here one day at a time.
3. **Weights.** v = the anchor's `w_reg` (√mcap scale), **renormalized over
   the day's realized sample** (`v = v / Σv`). No `w_reg = √mcap/Σ√mcap`
   re-assert and no mcap cross-check: the anchor's own build already
   guarantees both, and `mcap` is never loaded here.
4. **Design.** Industries with zero members that day are excluded from the
   design and get `f = null`; style columns identically zero that day are
   dropped and get `f = null` — same conventions as monthly. Constraint
   weights are the **full-ESTU cap weights at t**, fixed for every day of the
   month and deliberately NOT renormalized to the day's sample (the monthly
   spec's decision carries over unchanged).
5. **Solve.** √v-scaled `lstsq`. If the reduced design is rank-deficient →
   **skip the day** and count it; do not hard-fail. The monthly build asserts
   full rank because one bad month among ~330 is a data problem worth
   stopping for; one bad day must not kill a 6,000-day build (master list #4).
6. **Outputs per solved day.** 68 factor rows (f; shared weighted R²,
   computed against the v-weighted mean return; shared n_stocks) plus the
   ESTU residual vector `u = y − X g` — one u per stock in the day's sample.
   These residuals ARE the daily specific returns; nothing more is computed.
   Keep the per-day condition number (from the lstsq singular values) and the
   recomputed constraint residual `Σ_j w_j·f_ind,j` as in-notebook
   diagnostics for §6 check 3.

**What the daily build deliberately does NOT do** (vs monthly): no non-ESTU
out-of-sample extension, no `w_reg`/mcap asserts, no near-constant
style-column assert (std > 1e-8), no per-transition drop diagnostics
(return-missing / exposure-missing / UNASSIGNED counters) — one skipped-day
counter replaces them all. The monthly build is the heavily-instrumented
canonical regression; the daily build is the same engine run 6,350 times with
only the guardrails that scale.

Reference run: 6,350 days solved, 524 thin days skipped (all early-1999/2000
history), mean daily R² 0.253, max condition number 49.9, max constraint
residual 1.3×10⁻¹⁸.

## §5. Monthly vs daily — every difference

The two builds share the engine; everything that legitimately differs is in
this table. If you find a difference not listed here, one of your two builds
has a bug.

| Aspect | Monthly (`csr_build.ipynb`) | Daily (`daily_csr_build.ipynb`) |
|---|---|---|
| Regression frequency | one WLS per month-transition (t, t+1] | one WLS per trading day; exposures fixed at the owning month-end |
| Target y | sum of daily excess log returns over the owned month | single-day excess log return |
| Non-ESTU extension | yes — coverage-universe residuals via out-of-sample fitted f (`in_estu = False` rows) | **none** — ESTU residuals only |
| Specific-return schema | `permaticker, signal_date, ret_date, in_estu, y, y_hat, u, w_reg` | `permaticker, date, u` (sorted `[permaticker, date]`) |
| Factor-return date columns | `signal_date` + `ret_date` | single `date` |
| Rank deficiency | hard assert — the build fails | day **skipped**, counted |
| Skip rule | return frame < 100 rows or complete-case n < 100 | complete-case n < 100 (524 days on the reference run) |
| `w_reg = √mcap/Σ√mcap` assert (1e-9) + mcap cross-check (1e-3) | yes | absent — anchor `w_reg` trusted, renormalized per day; `mcap` never loaded |
| Near-constant style assert (std > 1e-8) | yes | absent (identically-zero columns still dropped, f = null) |
| Drop diagnostics | per-transition counters: return-missing, exposure-missing, UNASSIGNED | single skipped-day counter |
| Validation | 8 checks (identities, ESTU orthogonality, market corr, constraint, conditioning, R², flag integrity, OOS spread) | 4 checks (§6) |
| Extra constant | — | reference build defined `RECONCILE_TOL = 1e-9`, unused — dropped here (§3) |
| Reference numbers | monthly CSR textbook chapter | none of its own — validated by reconciliation to the monthly CSR (§6) |

Everything absent from this table — anchor, exposure joins, constraint
weights, reparametrization, eliminated-industry choice, UNASSIGNED handling,
complete-case stance — is identical to the monthly spec by construction.

## §6. STAGE 6 — Validation contract

🧪 Four checks. Reference-run observed values are given: your rebuild should
land near them (tail-vintage drift aside). A check that barely scrapes its
gate is a red flag even when it passes — every observed value below clears
its gate by a wide margin.

| check | gate | reference run |
|---|---|---|
| 1 | corr(daily f_country, daily cap-weighted ESTU market excess return) > 0.95 | **0.9979** |
| 2 | reconciliation: per (owning month-end, factor), Σ of intramonth daily f vs the monthly CSR f — corr > 0.95 | corr **0.9919**, RMSE **0.46pp** |
| 3 | well-posed **every day**: max condition number of the scaled design < 1e6 AND max daily \|Σ_j w_j·f_ind,j\| < 1e-8 | cond **49.9**, resid **1.3×10⁻¹⁸** |
| 4 | coverage: > 6,000 trading days carry factor returns | **6,350** |

Notes:

- **Check 1** is the USE4 signature at daily frequency (the monthly analogue
  gates at 0.99). The daily ESTU market excess return comes from the same
  shared panel (`mkt_ret`), restricted to owned days.
- **Check 2** is the joint integrity test of BOTH builds — it can only pass
  if the ownership mapping partitions the calendar identically in the two
  notebooks. Join on (owning month-end, factor), drop null f, sum the daily f
  within each month, correlate against the monthly f, and report the RMSE.
  It is statistical, not exact (§3) — but corr well below the reference or
  RMSE ≫ 0.5pp means the mappings disagree, not that "daily is noisy".
  Requires `data/out/csr_factor_returns.parquet` — run the monthly build
  first.
- **Check 3** re-verifies conditioning and the constraint from the in-loop
  diagnostics (§4 step 6): condition number from the lstsq singular values;
  constraint residual recomputed from the recovered f including the
  eliminated industry. Observed values sit four and ten orders of magnitude
  inside their gates.
- **Check 4** guards against silent mass-skipping: 6,350 solved of 6,874
  candidate days; the 524 skips are all thin early-1999/2000 history. Any
  skip after 2000 → investigate before shipping.

Informational (print, no gate): mean daily R² (reference 0.253 — the monthly
mean is 0.252; they should be close), residual-file stock-day and ticker
counts, skipped-day count, per-day sample-size distribution.

## §7. Master list of ❓ NOT-IN-PDF decisions (daily-specific)

The engine decisions — constraint implementation, eliminated industry, return
convention, degenerate factor-dates, UNASSIGNED handling, RF publication lag,
constraint weights vs realized sample — live in the monthly spec's master
list (`csr_spec.ipynb` §6) and carry over unchanged. These are the decisions
the daily clock adds:

| # | Decision | Default chosen | Alternative | Where to revisit |
|---|---|---|---|---|
| 1 | Intramonth exposures | Month-end exposures held fixed; day d explained by the latest month-end strictly before d | Re-standardize / interpolate exposures daily | Only if the whole pipeline moves to a daily rebalance — that is a different model, not a flag |
| 2 | Day→month ownership | Previous-trading-day as-of (backward) mapping, identical to the monthly build's owned-days mapping | Calendar-month membership | Never — identical mappings are what make §6 check 2 meaningful: both builds sum exactly the same days |
| 3 | Thin days | Skip days with complete-case n < 100 (same floor as monthly); 524 skipped on the reference run, all early history | Lower the floor to extend the daily history | If a downstream estimator ever needs the early-1999 daily history — today none does |
| 4 | Rank-deficient days | Skip and count — never hard-fail | Monthly-style hard assert | If a skip ever lands outside the thin early history — that is an upstream data problem, not noise |
| 5 | Residual universe | ESTU-only residuals | Extend to the coverage universe like the monthly file | If the specific-risk time-series layers ever need daily non-ESTU residuals; today coverage-universe treatment rides on the monthly extension |
| 6 | Specific-return schema | `permaticker, date, u` only, sorted `[permaticker, date]` | Carry y, ŷ, w_reg and flags like the monthly file | If re-audit convenience ever outweighs lean storage — at 13.4M rows, u is all the specific-risk build reads |
| 7 | Single `date` column | The factor file keys on the return day only; recover the owning month-end via the §2 mapping when needed | Carry `signal_date` + `ret_date` like the monthly file | Never — the covariance build keys on the return day; duplicating the mapping into the schema invites drift |
| 8 | Reconciliation tolerance | Statistical reconciliation (corr > 0.95, RMSE reported); the reference build's unused exact `RECONCILE_TOL = 1e-9` dropped | Enforce Σ daily f = monthly f exactly | Never — per-day complete-case samples differ from the pooled monthly sample, so exact equality is impossible by construction |
| 9 | Daily validation gates | corr gates at 0.95 (monthly: 0.99); per-day constraint gate 1e-8 (monthly: 1e-10); no daily R² gate | Reuse the monthly gates verbatim | If observed values (0.9979 / 0.9919 / 1.3×10⁻¹⁸) ever drift toward the gates |

**Reference, restated.** The daily CSR has no textbook chapter of its own; the
monthly CSR chapter carries the reference numbers, and the §6 contract
reconciles this build against them.